In [5]:
import pandas as pd
import numpy as np

# 1. Cargar el dataset procesado
path = '../data/processed/aves_procesado_markov.csv'
df = pd.read_csv(path)

# Aseguramos que la fecha sea datetime y ordenamos
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['trayectoria_id', 'date'])

print(f"Dataset cargado: {df.shape[0]} registros listos para Markov.")

Dataset cargado: 22041 registros listos para Markov.


In [6]:
import pandas as pd
import numpy as np

# 1. Cargar datos
df_final = pd.read_csv('../data/processed/aves_procesado_markov.csv')
df_final['date'] = pd.to_datetime(df_final['date'])

# 2. Configuración del Grid
res = 0.5

# Usamos el valor mínimo global para que el (0,0) del grid sea consistente
lon_min = df_final['lon'].min()
lat_min = df_final['lat'].min()

# 3. Calcular coordenadas del grid
# Restamos el mínimo y dividimos por la resolución para obtener el índice de celda
df_final['grid_x'] = ((df_final['lon'] - lon_min) / res).astype(int)
df_final['grid_y'] = ((df_final['lat'] - lat_min) / res).astype(int)

# 4. Crear el ID de celda (Estado del modelo de Markov)
df_final['cell_id'] = df_final['grid_x'].astype(str) + "_" + df_final['grid_y'].astype(str)

print(f"Celdas únicas generadas con resolución {res}: {df_final['cell_id'].nunique()}")

Celdas únicas generadas con resolución 0.5: 1253


In [7]:
# 1. Ordenar por trayectoria y fecha para asegurar saltos lógicos
df_final = df_final.sort_values(['trayectoria_id', 'date'])

# 2. Obtener la celda del día siguiente dentro de cada trayectoria
df_final['next_cell_id'] = df_final.groupby('trayectoria_id')['cell_id'].shift(-1)

# 3. Crear el dataset de transiciones (eliminando el último día de cada trayectoria)
df_markov = df_final.dropna(subset=['next_cell_id']).copy()

print(f"Dataset de transiciones listo: {len(df_markov)} registros.")
print(df_markov[['trayectoria_id', 'date', 'cell_id', 'next_cell_id']].head())

Dataset de transiciones listo: 21561 registros.
    trayectoria_id       date cell_id next_cell_id
309   91732A_Seg10 2010-04-18   49_63        49_64
310   91732A_Seg10 2010-04-19   49_64        49_64
311   91732A_Seg10 2010-04-20   49_64        49_64
312   91732A_Seg10 2010-04-21   49_64        49_64
313   91732A_Seg10 2010-04-22   49_64        49_64


In [8]:
import pandas as pd

# 1. Aseguramos que los datos estén ordenados por trayectoria y fecha
df_markov = df_markov.sort_values(['trayectoria_id', 'date'])

# 2. Extraer el mes numérico de la fecha
df_markov['mes_num'] = pd.to_datetime(df_markov['date']).dt.month

# Diccionario para guardar las 12 matrices
matrices_mensuales = {}

# Nombres de los meses para los prints
nombres_meses = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril", 
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto", 
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}

# 3. Bucle para generar cada matriz
for mes in range(1, 13):
    # Filtrar datos del mes actual
    df_mes = df_markov[df_markov['mes_num'] == mes]
    
    if not df_mes.empty:
        # Agrupar por celda origen y celda destino para contar frecuencias
        # Usamos next_cell_id que ya calculamos previamente con el groupby(trayectoria_id)
        matriz = df_mes.groupby(['cell_id', 'next_cell_id']).size().reset_index(name='frecuencia')
        
        # Calcular la probabilidad de transición
        # Sumamos todas las salidas desde cada celda de origen en este mes
        suma_origen = matriz.groupby('cell_id')['frecuencia'].transform('sum')
        matriz['probabilidad'] = matriz['frecuencia'] / suma_origen
        
        # Guardar en el diccionario
        matrices_mensuales[mes] = matriz.sort_values(['cell_id', 'probabilidad'], ascending=[True, False])
        
        print(f"Matriz de {nombres_meses[mes]} creada. Transiciones únicas: {len(matriz)}")
    else:
        print(f"Advertencia: No hay datos para el mes {nombres_meses[mes]}")

# --- Ejemplo de uso ---
# Para ver la matriz de Septiembre (mes con muchos datos según tus gráficas):
print("\nEjemplo de transiciones en Septiembre:")
print(matrices_mensuales[9].head(10))

Matriz de Enero creada. Transiciones únicas: 102
Matriz de Febrero creada. Transiciones únicas: 98
Matriz de Marzo creada. Transiciones únicas: 171
Matriz de Abril creada. Transiciones únicas: 370
Matriz de Mayo creada. Transiciones únicas: 213
Matriz de Junio creada. Transiciones únicas: 105
Matriz de Julio creada. Transiciones únicas: 119
Matriz de Agosto creada. Transiciones únicas: 490
Matriz de Septiembre creada. Transiciones únicas: 995
Matriz de Octubre creada. Transiciones únicas: 693
Matriz de Noviembre creada. Transiciones únicas: 347
Matriz de Diciembre creada. Transiciones únicas: 183

Ejemplo de transiciones en Septiembre:
  cell_id next_cell_id  frecuencia  probabilidad
0   0_110        6_101           1           1.0
1   10_87        12_86           1           1.0
2   10_92        11_92           1           0.5
3   10_92         8_90           1           0.5
4   10_97        11_95           1           1.0
5  11_113       11_113           1           0.5
6  11_113    